安全除法

写一个函数 safe_divide(a, b)，返回除法结果。如果除数为零，返回 None；如果参数类型不对，打印错误信息并返回 None。

```
# 期望行为
safe_divide(10, 2)    # 5.0
safe_divide(10, 0)    # None
safe_divide("a", 2)   # None（同时打印 "类型错误" 提示）
```

In [1]:
import json
from random import random
from time import sleep

from urllib3.exceptions import MaxRetryError


def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None
    except TypeError:
        print("类型错误")
        return None


# Expected behavior
print(safe_divide(10, 2))    # 5.0
print(safe_divide(10, 0))    # None
print(safe_divide("a", 2))   # prints 类型错误, returns None


5.0
None
类型错误
None


JSON 安全解析

写一个函数 parse_config(json_str: str) -> dict。如果 JSON 格式错误，打印错误信息并返回空字典 {}。用 try/except/else 结构：只有解析成功时才打印 “配置加载成功”。

In [ ]:
import json


def parse_config(json_str: str) -> dict:
    """Parse a JSON config string.

    Return {} when the JSON is malformed; print a status message either way.
    """
    try:
        config = json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"JSON 格式错误: {e}")
        return {}
    else:
        print("配置加载成功")
        return config


# Expected behavior
print(parse_config('{"name": "agent", "debug": true}'))   # 配置加载成功 + dict
print(parse_config('{"name": }'))                          # JSON 格式错误 + {}
print(parse_config('not json at all'))                     # JSON 格式错误 + {}


文件按行读取

写一个函数 read_lines_safe(filepath: str) -> list[str]，读取文件的每一行。文件不存在时返回空列表；读取过程中发生任何错误时，确保文件被关闭。

In [ ]:
def read_lines_safe(filepath: str) -> list[str]:
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            return f.readlines()
    except FileNotFoundError:
        return []

print(read_lines_safe("/Users/yiqi/project/agentStudy/课前预习/01-第一章-Python工程速通/demo/README2.md"))

自定义异常 + 异常链

定义 APIError 异常类，包含 status_code 和 response_body 属性。然后写一个函数：

* 调用一个模拟的 API（你可以用 raise 来模拟 API 错误）
* 如果 API 返回非 2xx，抛出自定义的 APIError
* 用 from e 保留原始异常链

In [ ]:
class APIError(Exception):
    def __init__(self, status_code: int, response_body: str):
        self.status_code = status_code
        self.response_body = response_body
        super().__init__()

def call_api():
    print("模拟调用api")
    raise APIError(201, "{}")


try:
    call_api()
except APIError as e:
    raise RuntimeError from e

多层异常处理

模拟一个 “Agent → Tool → HTTP” 三层调用栈。在 HTTP 层抛出 ConnectionError，在 Tool 层用 RuntimeError 包装它（带 from e），在 Agent 层分别捕获 RuntimeError 和 ToolNotFoundError，输出不同的提示信息。

In [ ]:
class ToolNotFoundError(Exception):
    pass

def http():
    raise ConnectionError("连接错误")

def tool():
    try:
        http()
    except ConnectionError as ce:
        raise RuntimeError("RuntimeError包") from ce


def agent():
    try:
        tool()
    except RuntimeError:
        raise
    except ToolNotFoundError:
        raise


agent()

完整的 API 调用函数（带重试 + 降级）

写一个 call_llm_with_fallback(messages: list[dict], primary_model: str, fallback_model: str) -> str 函数：

* 先尝试调用主模型（primary_model），最多重试 2 次
* 如果主模型失败，自动降级到备用模型（fallback_model），备用模型也最多重试 1 次
* 如果两个模型都失败，返回一个友好的错误消息字符串（而不是抛异常让服务崩溃）
* 区分以下场景并给出不同日志：
    * 超时 → “模型 X 响应超时”
    * 限流（429） → “模型 X 被限流，等待 N 秒”
    * 其他错误 → “模型 X 未知错误：{详情}”

提示：你可以用装饰器或手动写重试循环。关键是要让主模型和备用模型形成双层保护。

In [3]:
import random
import time


class RateLimitError(Exception):
    """Simulated HTTP 429, carries the server-advised wait seconds."""

    def __init__(self, wait_seconds: int):
        super().__init__(f"HTTP 429, retry after {wait_seconds}s")
        self.wait_seconds = wait_seconds


def retry(max_attempts: int, delay: float):
    """Retry decorator with classified failure logs.

    Assumes the decorated function takes the model name as its first
    argument, so every log line can name the model.
    """
    def decorator(func):
        def wrapper(model: str, *args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(model, *args, **kwargs)
                except Exception as e:
                    # classify every failure, including the final one
                    if isinstance(e, TimeoutError):
                        print(f"模型 {model} 响应超时")
                    elif isinstance(e, RateLimitError):
                        print(f"模型 {model} 被限流，等待 {e.wait_seconds} 秒")
                    else:
                        print(f"模型 {model} 未知错误：{e}")
                    if attempt == max_attempts:
                        raise  # final failure: hand over to the fallback layer
                    # honor the 429-advised wait, otherwise the fixed delay
                    time.sleep(getattr(e, "wait_seconds", delay))
        return wrapper
    return decorator


def call_model(model: str, messages: list[dict]) -> str:
    """Mock LLM call that randomly fails to exercise every path."""
    outcome = random.random()
    if outcome < 0.3:
        raise TimeoutError("request timed out")
    if outcome < 0.5:
        raise RateLimitError(wait_seconds=2)
    if outcome < 0.6:
        raise ValueError("malformed response")
    return f"{model} 回复：已收到 {len(messages)} 条消息"


# Decorator is just syntax sugar: retry(...)(call_model) == @retry(...)
call_primary = retry(max_attempts=3, delay=1)(call_model)   # 1 call + up to 2 retries
call_fallback = retry(max_attempts=2, delay=1)(call_model)  # 1 call + up to 1 retry


def call_llm_with_fallback(messages: list[dict], primary_model: str, fallback_model: str) -> str:
    try:
        return call_primary(primary_model, messages)
    except Exception:
        print(f"主模型 {primary_model} 连续失败，降级到备用模型 {fallback_model}")

    try:
        return call_fallback(fallback_model, messages)
    except Exception:
        return f"抱歉，{primary_model} 和 {fallback_model} 均不可用，请稍后再试。"

call_llm_with_fallback([], "主模型", "备用模型")

模型 主模型 被限流，等待 2 秒
模型 主模型 响应超时
模型 主模型 未知错误：malformed response
主模型 主模型 连续失败，降级到备用模型 备用模型
模型 备用模型 被限流，等待 2 秒
模型 备用模型 被限流，等待 2 秒


'抱歉，主模型 和 备用模型 均不可用，请稍后再试。'